# 🚀 Hybrid RAG Pipeline - TriviaQA

**Dense (FAISS) + BM25 (rank_bm25) + Reranking + LLM Generation**

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order


## 1. Install Dependencies


In [ ]:
!pip install -q torch transformers datasets accelerate bitsandbytes rank_bm25 scikit-learn tqdm faiss-cpu sentence-transformers


## 2. Upload Project Files

Upload the `src/` folder to Colab (or clone from GitHub)


In [ ]:
# Option A: Upload files manually
from google.colab import files
import os
import zipfile

# Upload a zip file containing your src folder
print("Upload your project as a ZIP file (containing src/ folder)")
uploaded = files.upload()

# Extract if zip
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"Extracted {filename}")

print("\\nFiles in current directory:")
print(os.listdir('.'))


In [ ]:
# Setup paths
import sys
sys.path.insert(0, '.')

# Verify src folder exists
import os
if os.path.exists('src'):
    print("src/ folder found!")
    print("Contents:", os.listdir('src'))
else:
    print("ERROR: src/ folder not found. Please upload it.")


## 3. Check GPU


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Generation will be slow!")


## 4. Build Indices (Dense + BM25)

⚠️ **This will take 5-15 minutes** depending on sample size


In [ ]:
# Configuration
SAMPLE_SIZE = 1000  # Use 1000 for quick test, None for full dataset (slow!)
FORCE_REBUILD = True  # Set to False to reuse existing indices


In [ ]:
from src.data_loader import load_trivia_qa
from src.retriever import DenseRetriever, SparseRetriever
from src.utils import prepare_corpus
import os

# Check if indices exist
dense_exists = os.path.exists("rag_index.faiss")
bm25_exists = os.path.exists("bm25_index.pkl")

print(f"Dense index exists: {dense_exists}")
print(f"BM25 index exists: {bm25_exists}")

if FORCE_REBUILD or not (dense_exists and bm25_exists):
    print("\\n📦 Loading TriviaQA dataset...")
    data = load_trivia_qa()
    train_data = data["train"]
    
    if SAMPLE_SIZE:
        print(f"Using sample of {SAMPLE_SIZE} examples")
        train_data = train_data.select(range(min(SAMPLE_SIZE, len(train_data))))
    
    print("\\n📄 Preparing corpus (chunking documents)...")
    corpus = prepare_corpus(train_data)
    print(f"Created {len(corpus)} chunks")
    
    # Build Dense Index
    print("\\n🔍 Building Dense Index (FAISS)...")
    dense_retriever = DenseRetriever()
    dense_retriever.build_index(corpus)
    dense_retriever.save_index("rag_index")
    print("✅ Dense index saved to rag_index.faiss")
    
    # Build BM25 Index  
    print("\\n🔍 Building BM25 Index (rank_bm25)...")
    sparse_retriever = SparseRetriever()
    sparse_retriever.build_index(corpus)
    sparse_retriever.save_index("bm25_index.pkl")
    print("✅ BM25 index saved to bm25_index.pkl")
    
    # Clear memory
    del corpus, train_data
    torch.cuda.empty_cache()

print("\\n✅ Indices ready!")


In [ ]:
from src.retriever import DenseRetriever
from src.re_ranker import Reranker
from src.generator import RAGGenerator
import pickle

# Load Dense Retriever
print("Loading Dense Retriever (FAISS)...")
dense_retriever = DenseRetriever()
dense_retriever.load_index("rag_index")
print(f"✅ Dense index: {len(dense_retriever.corpus)} documents")

# Load BM25
print("\\nLoading BM25 Index...")
with open("bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data["bm25"]
bm25_corpus = bm25_data["corpus"]
print(f"✅ BM25 index: {len(bm25_corpus)} documents")

# Load Reranker
print("\\nLoading Reranker (CrossEncoder)...")
reranker = Reranker()
print("✅ Reranker loaded")

# Load Generator
print("\\nLoading LLM (TinyLlama)...")
generator = RAGGenerator()
print("✅ Generator loaded")


In [ ]:
def hybrid_rag(query, top_k_retrieval=20, top_k_rerank=5, verbose=True):
    """
    Full Hybrid RAG Pipeline:
    1. Dense Retrieval (FAISS)
    2. BM25 Retrieval (rank_bm25)
    3. Combine & Deduplicate
    4. Rerank (CrossEncoder)
    5. Generate (LLM)
    """
    if verbose:
        print(f"\\n{'='*60}")
        print(f"🔍 Query: {query}")
        print(f"{'='*60}")
    
    # 1. Dense Retrieval
    dense_docs = dense_retriever.retrieve(query, top_k=top_k_retrieval)
    dense_texts = [doc["text"] for doc in dense_docs]
    if verbose:
        print(f"📊 Dense: {len(dense_texts)} docs")
    
    # 2. BM25 Retrieval
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_ids = bm25_scores.argsort()[-top_k_retrieval:][::-1]
    bm25_texts = [bm25_corpus[i]["text"] for i in top_bm25_ids if i < len(bm25_corpus)]
    if verbose:
        print(f"📊 BM25: {len(bm25_texts)} docs")
    
    # 3. Combine & Deduplicate
    combined = list(set(dense_texts + bm25_texts))
    if verbose:
        print(f"📊 Combined: {len(combined)} docs")
    
    # 4. Rerank
    contexts = reranker.rerank(query, combined, top_k=top_k_rerank)
    if verbose:
        print(f"📊 Reranked: {len(contexts)} docs")
    
    # 5. Generate
    context_str = "\\n\\n".join(contexts)
    answer = generator.generate_answer(query, context_str)
    
    if verbose:
        print(f"\\n💡 Answer: {answer}")
        print(f"{'='*60}")
    
    return answer


## 7. Test Single Query


In [ ]:
# Test with a sample question
query = "Who wrote Romeo and Juliet?"
answer = hybrid_rag(query)


In [ ]:
# Try more questions!
test_questions = [
    "Who painted the Mona Lisa?",
    "What is the capital of France?",
    "When did World War II end?",
    "Who discovered penicillin?"
]

for q in test_questions:
    answer = hybrid_rag(q, verbose=False)
    print(f"Q: {q}")
    print(f"A: {answer}\\n")


## 8. Batch Evaluation (Optional)


In [ ]:
from tqdm.notebook import tqdm
from src.data_loader import load_trivia_qa
from src.evaluation import evaluate_predictions
import json

# Load validation set
EVAL_SAMPLE_SIZE = 50  # Adjust for speed vs coverage

print("Loading validation set...")
data = load_trivia_qa()
val_data = data["validation"].select(range(EVAL_SAMPLE_SIZE))

predictions = []

for example in tqdm(val_data, desc="Evaluating"):
    question = example["question"]
    q_id = example["question_id"]
    
    # Run Hybrid RAG
    answer = hybrid_rag(question, verbose=False)
    
    predictions.append({
        "id": q_id,
        "question": question,
        "prediction": answer,
        "answers": example["answer"]["aliases"]
    })

# Save predictions
with open("predictions_colab.json", "w") as f:
    json.dump(predictions, f, indent=2)

print(f"\\n✅ Saved {len(predictions)} predictions")


In [ ]:
# Evaluate
preds_dict = {p["id"]: p["prediction"] for p in predictions}
refs_dict = {p["id"]: p["answers"] for p in predictions}

metrics = evaluate_predictions(preds_dict, refs_dict)

print("\\n" + "="*40)
print("📊 EVALUATION RESULTS")
print("="*40)
print(f"Exact Match: {metrics['exact_match']:.2f}%")
print(f"F1 Score:    {metrics['f1']:.2f}%")
print(f"Total:       {metrics['total']}")
print("="*40)


## 9. Interactive Mode (Ask Your Own Questions!)


In [ ]:
# Enter your own question here!
my_question = "What year was the Eiffel Tower built?"

answer = hybrid_rag(my_question)
